# 06 — Visualisasi & Inference
**Project**: YOLOv11s Person Detection  
**Tujuan**: Inference pada gambar baru + export model

---

## 6.1 Import Libraries

In [ ]:
import torch
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

sns.set_theme(style='whitegrid')

# Load model terbaik
with open('runs/training_summary.json') as f:
    train_summary = json.load(f)

BEST_MODEL_PATH = train_summary['model_path']
DEVICE          = 0 if torch.cuda.is_available() else 'cpu'
CONF_THRESH     = 0.25
IOU_THRESH      = 0.5
IMGSZ           = 640

print(f'Model   : {BEST_MODEL_PATH}')
print(f'Device  : {DEVICE}')
print(f'Conf    : {CONF_THRESH}')
print(f'Best mAP@0.5: {train_summary["best_map50"]:.4f}')

## 6.2 Load Model

In [ ]:
model = YOLO(BEST_MODEL_PATH)
print(f'Model berhasil di-load!')
model.info()

## 6.3 Siapkan Gambar Baru untuk Inference

Gunakan gambar dari **luar** dataset COCO 2017 — gambar yang belum pernah dilihat model sama sekali.

In [ ]:
import urllib.request
import ssl

INFERENCE_DIR = Path('inference_images')
INFERENCE_DIR.mkdir(exist_ok=True)

# Opsi 1: Gunakan gambar dari COCO test set (tidak ada ground truth publik)
# COCO test set images bisa diunduh dari: http://images.cocodataset.org/test2017/

# Opsi 2: Unduh gambar sample dari URL publik
# Di bawah ini beberapa gambar publik yang bisa digunakan untuk demo
SAMPLE_URLS = [
    # Gunakan URL gambar publik berlisensi bebas
    # Contoh format (ganti dengan URL aktual yang valid):
    # 'https://upload.wikimedia.org/wikipedia/commons/thumb/...',
]

# Opsi 3 (RECOMMENDED): Taruh gambar sendiri di folder 'inference_images/'
# Format yang didukung: .jpg, .jpeg, .png
print('Cara menyiapkan gambar baru:')
print('  1. Taruh file gambar (.jpg/.png) di folder: inference_images/')
print('  2. Atau ubah SAMPLE_URLS di atas dengan URL gambar publik')
print(f'\nFolder inference: {INFERENCE_DIR.resolve()}')

# Jika tidak ada gambar, gunakan beberapa gambar dari test set sebagai fallback
test_images_available = list(Path('dataset/images/test').glob('*.jpg'))
inference_images      = list(INFERENCE_DIR.glob('*.jpg')) + \
                        list(INFERENCE_DIR.glob('*.jpeg')) + \
                        list(INFERENCE_DIR.glob('*.png'))

if not inference_images:
    print('\nFolder inference kosong — menggunakan 12 gambar dari test set sebagai demo')
    import random, shutil
    random.seed(42)
    sample_test = random.sample(test_images_available, min(12, len(test_images_available)))
    for img in sample_test:
        shutil.copy(img, INFERENCE_DIR / img.name)
    inference_images = list(INFERENCE_DIR.glob('*.jpg'))

print(f'\nJumlah gambar untuk inference: {len(inference_images)}')

## 6.4 Jalankan Inference

In [ ]:
OUTPUT_DIR = Path('runs/inference/person_detection')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Menjalankan inference pada {len(inference_images)} gambar...')

results = model.predict(
    source    = [str(p) for p in inference_images],
    conf      = CONF_THRESH,
    iou       = IOU_THRESH,
    imgsz     = IMGSZ,
    device    = DEVICE,
    save      = True,
    save_conf = True,
    project   = 'runs/inference',
    name      = 'person_detection',
    exist_ok  = True,
    verbose   = False
)

print(f'\nInference selesai!')
print(f'Hasil disimpan di: runs/inference/person_detection/')

## 6.5 Visualisasi Grid: Sebelum vs Sesudah

In [ ]:
def draw_detections(img_path, result, conf_thresh=0.25):
    """Gambar bounding box deteksi di atas gambar."""
    img = np.array(Image.open(img_path).convert('RGB'))
    overlay = img.copy()
    h, w = img.shape[:2]

    if result.boxes is None or len(result.boxes) == 0:
        return img, 0

    count = 0
    for box in result.boxes:
        conf = float(box.conf[0])
        if conf < conf_thresh:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())

        # Warna berdasarkan confidence (hijau = tinggi, kuning = sedang)
        color_intensity = int(conf * 255)
        color = (0, color_intensity, 255 - color_intensity)

        # Bbox
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

        # Label background
        label = f'person {conf:.2f}'
        (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(overlay, (x1, y1 - lh - 6), (x1 + lw + 4, y1), color, -1)
        cv2.putText(overlay, label, (x1 + 2, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        count += 1

    # Jumlah deteksi di pojok kiri atas
    cv2.putText(overlay, f'{count} person detected',
                (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)

    return overlay, count


# Pilih 9 gambar pertama untuk visualisasi
n_show      = min(9, len(inference_images))
sample_imgs = inference_images[:n_show]
sample_results = results[:n_show]

n_cols = 3
n_rows = (n_show + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Hasil Inference YOLOv11s — Person Detection\n'
             'Warna bbox: hijau = confidence tinggi, kuning = sedang',
             fontsize=13)

for idx, (img_path, result) in enumerate(zip(sample_imgs, sample_results)):
    row, col = divmod(idx, n_cols)
    ax = axes[row][col]

    annotated, count = draw_detections(img_path, result)
    ax.imshow(annotated)
    ax.set_title(f'{img_path.name}\n{count} person terdeteksi', fontsize=9)
    ax.axis('off')

# Sembunyikan axes yang tidak terpakai
for idx in range(n_show, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row][col].axis('off')

plt.tight_layout()
out_path = 'runs/inference/person_detection/inference_grid.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Grid visualisasi disimpan ke: {out_path}')

## 6.6 Benchmark Kecepatan Inference (FPS & Latency)

In [ ]:
import time

N_WARMUP = 5
N_BENCH  = 30
bench_img = inference_images[0]

# Warmup
print(f'Warmup ({N_WARMUP} run)...')
for _ in range(N_WARMUP):
    _ = model.predict(source=str(bench_img), conf=CONF_THRESH, iou=IOU_THRESH,
                      imgsz=IMGSZ, device=DEVICE, verbose=False)

# Benchmark
print(f'Benchmark ({N_BENCH} run)...')
latencies = []
for _ in range(N_BENCH):
    t0 = time.perf_counter()
    _  = model.predict(source=str(bench_img), conf=CONF_THRESH, iou=IOU_THRESH,
                       imgsz=IMGSZ, device=DEVICE, verbose=False)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)  # ms

latencies = np.array(latencies)
mean_lat  = latencies.mean()
std_lat   = latencies.std()
fps       = 1000 / mean_lat

print('\n=== Benchmark Inference ===')
print(f'  Device          : {"GPU" if torch.cuda.is_available() else "CPU"}')
print(f'  imgsz           : {IMGSZ}')
print(f'  Mean latency    : {mean_lat:.2f} ms ± {std_lat:.2f} ms')
print(f'  Min latency     : {latencies.min():.2f} ms')
print(f'  Max latency     : {latencies.max():.2f} ms')
print(f'  Throughput      : {fps:.1f} FPS')
print('===========================')

# Plot latency distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(mean_lat, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_lat:.1f} ms')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Frekuensi')
ax.set_title(f'Distribusi Latency Inference | {fps:.1f} FPS')
ax.legend()
plt.tight_layout()
plt.savefig('runs/inference/person_detection/latency_benchmark.png', dpi=100)
plt.show()

## 6.7 Dashboard Statistik Inference

In [ ]:
# Kumpulkan statistik dari semua inference
all_detections = []
for img_path, result in zip(inference_images, results):
    n_det = len(result.boxes) if result.boxes is not None else 0
    confs = result.boxes.conf.cpu().numpy().tolist() if result.boxes and n_det > 0 else []
    all_detections.append({
        'image'   : img_path.name,
        'n_person': n_det,
        'confs'   : confs,
        'max_conf': max(confs) if confs else 0,
        'avg_conf': np.mean(confs) if confs else 0,
    })

df_det = pd.DataFrame(all_detections)
all_confs = [c for row in all_detections for c in row['confs']]

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Dashboard Statistik Inference — Person Detection', fontsize=14, y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

# 1. Distribusi jumlah deteksi per gambar
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df_det['n_person'], bins=range(0, df_det['n_person'].max()+2),
         color='steelblue', edgecolor='white', linewidth=0.5, align='left')
ax1.set_xlabel('Jumlah person terdeteksi')
ax1.set_ylabel('Jumlah gambar')
ax1.set_title('Distribusi Deteksi per Gambar')
ax1.axvline(df_det['n_person'].mean(), color='red', linestyle='--',
            label=f'Avg: {df_det["n_person"].mean():.1f}')
ax1.legend(fontsize=8)

# 2. Distribusi confidence score
ax2 = fig.add_subplot(gs[0, 1])
if all_confs:
    ax2.hist(all_confs, bins=30, color='green', edgecolor='white', linewidth=0.5)
    ax2.axvline(np.mean(all_confs), color='red', linestyle='--',
                label=f'Mean: {np.mean(all_confs):.3f}')
    ax2.axvline(CONF_THRESH, color='orange', linestyle=':', label=f'Threshold: {CONF_THRESH}')
    ax2.legend(fontsize=8)
ax2.set_xlabel('Confidence Score')
ax2.set_ylabel('Jumlah deteksi')
ax2.set_title('Distribusi Confidence Score')

# 3. Top gambar (jumlah deteksi terbanyak)
ax3 = fig.add_subplot(gs[0, 2])
top_imgs = df_det.nlargest(8, 'n_person')
bars = ax3.barh(range(len(top_imgs)), top_imgs['n_person'].values, color='purple', alpha=0.8)
ax3.set_yticks(range(len(top_imgs)))
ax3.set_yticklabels([name[:20] for name in top_imgs['image'].values], fontsize=7)
ax3.set_xlabel('Jumlah person terdeteksi')
ax3.set_title('Top Gambar: Deteksi Terbanyak')

# 4. Metrik summary
ax4 = fig.add_subplot(gs[1, :])
ax4.axis('off')

with open('runs/eval_results.json') as f:
    eval_res = json.load(f)

summary_data = [
    ['mAP@0.5',         f"{eval_res['test_map50']:.4f}",   'Target: ≥ 0.60',
     '✓ TERCAPAI' if eval_res['test_map50'] >= 0.60 else '✗ BELUM'],
    ['mAP@0.5:0.95',    f"{eval_res['test_map5095']:.4f}", '—', '—'],
    ['Precision',       f"{eval_res['test_precision']:.4f}", '—', '—'],
    ['Recall',          f"{eval_res['test_recall']:.4f}",   '—', '—'],
    ['F1 Score',        f"{eval_res['test_f1']:.4f}",       '—', '—'],
    ['Avg confidence',  f"{np.mean(all_confs):.4f}" if all_confs else '—', f'Threshold: {CONF_THRESH}', '—'],
    ['Inference FPS',   f'{fps:.1f}', f'{mean_lat:.1f} ms/gambar', '—'],
    ['Total gambar uji',f'{len(inference_images)}', '—', '—'],
    ['Avg deteksi/img', f"{df_det['n_person'].mean():.1f}", '—', '—'],
]

table = ax4.table(
    cellText   = summary_data,
    colLabels  = ['Metrik', 'Nilai', 'Keterangan', 'Status'],
    cellLoc    = 'center',
    loc        = 'center',
    bbox       = [0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#ecf0f1')
    cell.set_edgecolor('#bdc3c7')
ax4.set_title('Ringkasan Metrik Proyek', fontsize=11, pad=10)

plt.savefig('runs/inference/person_detection/inference_dashboard.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('Dashboard tersimpan ke: runs/inference/person_detection/inference_dashboard.png')

## 6.8 Visualisasi Detail: 1 Gambar Terbaik

In [ ]:
# Pilih gambar dengan deteksi terbanyak untuk visualisasi detail
best_idx  = df_det['n_person'].idxmax()
best_img  = inference_images[best_idx]
best_res  = results[best_idx]

img_orig = np.array(Image.open(best_img).convert('RGB'))
img_ann, n_det = draw_detections(best_img, best_res)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Visualisasi Detail: {best_img.name}', fontsize=12)

axes[0].imshow(img_orig)
axes[0].set_title('Gambar Original', fontsize=11)
axes[0].axis('off')

axes[1].imshow(img_ann)
axes[1].set_title(f'Hasil Deteksi: {n_det} person terdeteksi\n'
                  f'Conf threshold: {CONF_THRESH}', fontsize=11)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('runs/inference/person_detection/best_detection.png', dpi=120, bbox_inches='tight')
plt.show()

# Detail setiap bbox
if best_res.boxes is not None and len(best_res.boxes) > 0:
    print(f'\nDetail {n_det} deteksi pada gambar terbaik:')
    print(f'{"No":>4} {"x1":>6} {"y1":>6} {"x2":>6} {"y2":>6} {"Conf":>8}')
    print('-' * 45)
    for i, box in enumerate(best_res.boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        conf = float(box.conf[0])
        print(f'{i+1:>4} {x1:>6} {y1:>6} {x2:>6} {y2:>6} {conf:>8.4f}')

## 6.9 Export Model

In [ ]:
print('Mengekspor model ke format ONNX dan TorchScript...')

# Export ke ONNX
onnx_path = model.export(
    format  = 'onnx',
    imgsz   = IMGSZ,
    dynamic = False,
    simplify= True,
    opset   = 12
)
print(f'ONNX model tersimpan: {onnx_path}')

# Export ke TorchScript
ts_path = model.export(
    format  = 'torchscript',
    imgsz   = IMGSZ,
)
print(f'TorchScript model tersimpan: {ts_path}')

In [ ]:
# Benchmark ONNX vs PyTorch
print('Membandingkan kecepatan PyTorch vs ONNX...')

bench_results = model.benchmark(
    imgsz  = IMGSZ,
    device = DEVICE,
    half   = False     # True jika GPU mendukung FP16
)

print('\nHasil benchmark:')
print(bench_results)

## 6.10 Ringkasan Proyek Lengkap

In [ ]:
with open('runs/baseline_results.json') as f:
    baseline = json.load(f)

with open('runs/eval_results.json') as f:
    final_eval = json.load(f)

print('=' * 65)
print('               RINGKASAN PROYEK LENGKAP')
print('=' * 65)
print(f'Model              : YOLOv11s (yolo11s.pt)')
print(f'Dataset            : COCO 2017 — class person')
print(f'Total data         : 3000 gambar (70/15/15 split)')
print()
print(f'{"Metrik":25s} {"Baseline":>12s} {"Fine-tuned":>12s} {"Delta":>10s}')
print('-' * 65)
print(f'{"mAP@0.5":25s} {baseline["map50"]:>12.4f} {final_eval["test_map50"]:>12.4f} {final_eval["test_map50"]-baseline["map50"]:>+10.4f}')
print(f'{"mAP@0.5:0.95":25s} {baseline["map5095"]:>12.4f} {final_eval["test_map5095"]:>12.4f} {final_eval["test_map5095"]-baseline["map5095"]:>+10.4f}')
print(f'{"Precision":25s} {baseline["precision"]:>12.4f} {final_eval["test_precision"]:>12.4f} {final_eval["test_precision"]-baseline["precision"]:>+10.4f}')
print(f'{"Recall":25s} {baseline["recall"]:>12.4f} {final_eval["test_recall"]:>12.4f} {final_eval["test_recall"]-baseline["recall"]:>+10.4f}')
print(f'{"F1 Score":25s} {"—":>12s} {final_eval["test_f1"]:>12.4f} {"—":>10s}')
print()
print(f'{"Inference FPS":25s} {fps:>12.1f}')
print(f'{"Latency per gambar":25s} {mean_lat:>12.1f} ms')
print()
print(f'Target mAP@0.5 ≥ 0.60: {"TERCAPAI ✓" if final_eval["test_map50"] >= 0.60 else "BELUM TERCAPAI ✗"}')
print()
print('Output yang dihasilkan:')
print(f'  Trained model (PT)    : {BEST_MODEL_PATH}')
print(f'  Exported model (ONNX) : {onnx_path}')
print(f'  Inference grid        : runs/inference/person_detection/inference_grid.png')
print(f'  Dashboard             : runs/inference/person_detection/inference_dashboard.png')
print(f'  IoU distribution      : runs/eval/test_eval/iou_distribution.png')
print(f'  Confidence analysis   : runs/eval/test_eval/confidence_analysis.png')
print('=' * 65)

---
## ✅ Proyek Selesai!

| Output | Lokasi |
|--------|--------|
| Model terbaik (.pt) | `runs/train/.../weights/best.pt` |
| Model ONNX | Sejajar dengan best.pt |
| Inference grid | `runs/inference/person_detection/inference_grid.png` |
| Dashboard metrik | `runs/inference/person_detection/inference_dashboard.png` |
| IoU distribution | `runs/eval/test_eval/iou_distribution.png` |
| Confidence dist. | `runs/eval/test_eval/confidence_analysis.png` |
| PR Curve | `runs/eval/test_eval/PR_curve.png` |
| Confusion Matrix | `runs/eval/test_eval/confusion_matrix.png` |